# 07. Complex Wishart MP-PCA — 3-Core Parallel

`02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb`에서 생성한
**5-echo noisy complex GRE**를 complex-valued 데이터 그대로 사용하고,
각 local patch의 Hermitian covariance eigenvalue에 대해
finite-size Complex Wishart 최대 고유값 분포를 이용하여
noise component를 분류하는 실험적 MP-PCA 방식이다.

이번 버전은 서버 CPU 3개를 활용하기 위해
**Noise 10%, 30%, 50%를 각각 독립 프로세스에서 동시에 처리**한다.

```text
Process 1 → Noise 10% Complex Wishart MP-PCA
Process 2 → Noise 30% Complex Wishart MP-PCA
Process 3 → Noise 50% Complex Wishart MP-PCA
```

실험 원칙:

- 입력: `(X, Y, Z, 5)` complex64
- Echoes = 5
- Patch radius = 3 → `7 × 7 × 7`
- Patch voxels = 343
- Features = 5 complex echoes
- `ALPHA = 0.99`
- **Brain mask는 denoising에 사용하지 않음**
- **전체 volume denoising**
- Brain mask는 최종 평가 ROI용으로만 저장
- `complex_wishart_mppca.py`의 `complex_wishart_mppca()` 사용


In [ ]:
import os
import time
import gc
from pathlib import Path
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt


### 환경설정

입력 noisy data:

`./data/noisy_data/`

- `qsm_siemens_monopolar_noisy_10.npz`
- `qsm_siemens_monopolar_noisy_30.npz`
- `qsm_siemens_monopolar_noisy_50.npz`

출력:

`./data/denoised/complex_wishart/`

서버 실행 시 `complex_wishart_mppca.py`가
`~/workspace`에서 import 가능해야 한다.

3-core 병렬화는 noise level 단위로 수행한다.
각 worker가 자기 `.npz` 파일을 직접 읽고 Complex Wishart MP-PCA를 수행한 뒤
자기 결과 `.mat` 파일을 직접 저장한다.


In [ ]:
NOISE_LEVELS = [10, 30, 50]

PATCH_R = 3
N_ECHO = 5
ALPHA = 0.99
N_WORKERS = 3

NOISY_DIR = Path("./data/noisy_data").resolve()
OUT_DIR = Path("./data/denoised/complex_wishart").resolve()

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Noisy directory :", NOISY_DIR)
print("Output directory:", OUT_DIR)
print("Echoes          :", N_ECHO)
print("Patch radius    :", PATCH_R)
print("Alpha           :", ALPHA)
print("Parallel workers:", N_WORKERS)
print(
    "Patch size      :",
    f"{2 * PATCH_R + 1} x {2 * PATCH_R + 1} x {2 * PATCH_R + 1}"
)


### 1. 입력 파일 확인

부모 프로세스에서는 세 noise level의 입력 파일 존재 여부만 확인한다.

각 worker가 자기 noise level의 `.npz` 파일을 직접 읽도록 하여
큰 complex 4D 배열을 프로세스 간에 전달하지 않는다.


In [ ]:
input_paths = {}

for noise_level in NOISE_LEVELS:
    noisy_path = (
        NOISY_DIR /
        f"qsm_siemens_monopolar_noisy_{noise_level}.npz"
    )

    if not noisy_path.exists():
        raise FileNotFoundError(
            f"Noise 파일을 찾을 수 없습니다: {noisy_path}\n"
            "먼저 02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb를 실행하세요."
        )

    input_paths[noise_level] = noisy_path
    print(f"{noise_level}% 입력 확인 -> {noisy_path}")

print("\n모든 입력 파일 확인 완료")


### 2. Complex Wishart MP-PCA worker

각 worker는 하나의 noise level 전체를 담당한다.

처리 순서:

1. noisy complex GRE `(X,Y,Z,5)` 로드
2. evaluation용 `mask_brain`은 별도로 보관
3. denoising에는 brain mask 대신 **전체가 True인 full-volume mask** 사용
4. Complex Wishart MP-PCA 수행
5. complex denoised result / sigma map / retained component map 저장

즉, 실제 denoising ROI는 brain 내부가 아니라 **전체 volume**이다.


In [ ]:
def run_complex_wishart_worker(
    noise_level,
    noisy_path_str,
    out_dir_str,
    patch_r,
    alpha,
    n_echo
):
    import os
    import time
    import numpy as np
    import scipy.io as sio
    from pathlib import Path

    from complex_wishart_mppca import complex_wishart_mppca

    start_time = time.time()

    noisy_path = Path(noisy_path_str)
    out_dir = Path(out_dir_str)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"[PID {os.getpid()}] START | Noise {noise_level}% | "
        f"file={noisy_path.name}",
        flush=True
    )

    data = np.load(noisy_path, allow_pickle=False)

    noisy_complex = data["noisy_complex"].astype(np.complex64)
    eval_mask = data["mask_brain"].astype(bool)

    if noisy_complex.ndim != 4:
        raise ValueError(
            f"{noise_level}% noisy_complex는 4D여야 합니다. "
            f"현재 shape={noisy_complex.shape}"
        )

    if noisy_complex.shape[-1] != n_echo:
        raise ValueError(
            f"{noise_level}% 데이터는 {n_echo} echoes를 예상합니다. "
            f"실제={noisy_complex.shape[-1]}"
        )

    if eval_mask.shape != noisy_complex.shape[:3]:
        raise ValueError(
            f"{noise_level}% mask shape {eval_mask.shape} != "
            f"spatial shape {noisy_complex.shape[:3]}"
        )

    # Brain mask는 denoising에 사용하지 않는다.
    # 함수가 mask 인자를 받는 구조이므로 all-True mask로 전체 volume을 지정한다.
    full_volume_mask = np.ones(
        noisy_complex.shape[:3],
        dtype=bool
    )

    noise_ratio = float(data["noise_ratio"])
    sigma_input = float(data["sigma"])
    seed = int(data["seed"])
    overall_mean = float(data["overall_mean"])

    echo_times = (
        data["echo_times"].astype(np.float64)
        if "echo_times" in data.files
        else np.full(n_echo, np.nan, dtype=np.float64)
    )

    field_strength = (
        float(data["field_strength"])
        if "field_strength" in data.files
        else np.nan
    )

    print(
        f"[PID {os.getpid()}] COMPLEX WISHART MP-PCA | "
        f"Noise {noise_level}% | shape={noisy_complex.shape} | "
        f"dtype={noisy_complex.dtype} | patch_radius={patch_r} | "
        f"alpha={alpha} | ROI=FULL_VOLUME",
        flush=True
    )

    deno_complex, sigma_map, ncomp_map = complex_wishart_mppca(
        noisy_complex,
        mask=full_volume_mask,
        patch_radius=patch_r,
        alpha=alpha,
        return_sigma=True,
        return_ncomps=True
    )

    deno_complex = np.asarray(deno_complex, dtype=np.complex64)
    sigma_map = np.asarray(sigma_map)
    ncomp_map = np.asarray(ncomp_map)

    if deno_complex.shape != noisy_complex.shape:
        raise ValueError(
            f"{noise_level}% denoised shape {deno_complex.shape} "
            f"!= input {noisy_complex.shape}"
        )

    if sigma_map.shape != noisy_complex.shape[:3]:
        raise ValueError(
            f"{noise_level}% sigma_map shape {sigma_map.shape} "
            f"!= spatial shape {noisy_complex.shape[:3]}"
        )

    if ncomp_map.shape != noisy_complex.shape[:3]:
        raise ValueError(
            f"{noise_level}% ncomp_map shape {ncomp_map.shape} "
            f"!= spatial shape {noisy_complex.shape[:3]}"
        )

    if not np.iscomplexobj(deno_complex):
        raise TypeError(
            f"{noise_level}% Wishart 결과가 complex가 아닙니다."
        )

    output_path = (
        out_dir /
        f"denoised_complex_wishart_{noise_level}_r{patch_r}_a{alpha}.mat"
    )

    save_data = {
        "denoised_complex": deno_complex,
        "sigma_map": sigma_map,
        "ncomp_map": ncomp_map,
        "mask_brain": eval_mask.astype(np.uint8),
        "noise_level": np.int32(noise_level),
        "noise_ratio": np.float32(noise_ratio),
        "sigma_input": np.float32(sigma_input),
        "seed": np.int32(seed),
        "overall_mean": np.float32(overall_mean),
        "patch_radius": np.int32(patch_r),
        "alpha": np.float64(alpha),
        "n_echo": np.int32(n_echo),
        "echo_times": np.asarray(echo_times, dtype=np.float64),
        "field_strength": np.float32(field_strength),
        "denoise_region": "full_volume",
        "mask_usage": "evaluation_only",
        "parallel_workers": np.int32(3),
        "method": "complex_wishart_mppca",
    }

    sio.savemat(
        output_path,
        save_data,
        do_compression=True
    )

    elapsed = time.time() - start_time

    mean_sigma_eval = float(np.mean(sigma_map[eval_mask]))
    mean_ncomp_eval = float(np.mean(ncomp_map[eval_mask]))

    print(
        f"[PID {os.getpid()}] DONE | Noise {noise_level}% | "
        f"{elapsed / 60:.2f} min | "
        f"mean_sigma(eval ROI)={mean_sigma_eval:.6g} | "
        f"mean_ncomp(eval ROI)={mean_ncomp_eval:.4f}",
        flush=True
    )

    return {
        "noise_level": int(noise_level),
        "output_path": str(output_path),
        "elapsed_sec": float(elapsed),
        "pid": int(os.getpid()),
        "mean_sigma_eval": mean_sigma_eval,
        "mean_ncomp_eval": mean_ncomp_eval,
    }


### 3. 10% / 30% / 50%를 3코어 동시 실행

Linux 서버에서 `fork` multiprocessing context를 사용한다.

```text
Worker 1 → Noise 10% Complex Wishart MP-PCA
Worker 2 → Noise 30% Complex Wishart MP-PCA
Worker 3 → Noise 50% Complex Wishart MP-PCA
```

서버 제출 조건:

- `-c 3`
- `OMP_NUM_THREADS=1`
- `OPENBLAS_NUM_THREADS=1`
- `MKL_NUM_THREADS=1`

따라서 3개 worker가 각각 한 CPU core를 중심으로 사용한다.


In [ ]:
parallel_start = time.time()
worker_results = {}

ctx = mp.get_context("fork")

print("=" * 70)
print("3-Core Parallel Complex Wishart MP-PCA 시작")
print(f"Noise levels : {NOISE_LEVELS}")
print(f"Workers      : {N_WORKERS}")
print(f"Echoes       : {N_ECHO}")
print(f"Patch radius : {PATCH_R}")
print(f"Alpha        : {ALPHA}")
print("=" * 70)

with ProcessPoolExecutor(
    max_workers=N_WORKERS,
    mp_context=ctx
) as executor:

    future_to_level = {
        executor.submit(
            run_complex_wishart_worker,
            noise_level,
            str(input_paths[noise_level]),
            str(OUT_DIR),
            PATCH_R,
            ALPHA,
            N_ECHO
        ): noise_level
        for noise_level in NOISE_LEVELS
    }

    for future in as_completed(future_to_level):
        noise_level = future_to_level[future]

        try:
            result = future.result()
            worker_results[noise_level] = result

            print(
                f"MAIN: {noise_level}% 완료 | "
                f"PID={result['pid']} | "
                f"{result['elapsed_sec'] / 60:.2f} min"
            )

        except Exception as e:
            print(
                f"MAIN: {noise_level}% FAILED -> {repr(e)}"
            )
            raise

parallel_elapsed = time.time() - parallel_start

print("\n" + "=" * 70)
print("모든 noise level Complex Wishart MP-PCA 병렬 처리 완료")
print(f"전체 wall time: {parallel_elapsed / 60:.2f} min")
print("=" * 70)

for noise_level in sorted(worker_results):
    print(worker_results[noise_level])


### 4. 저장 결과 확인

각 worker가 자기 noise level의 결과를 직접 저장한다.

출력 파일:

- `denoised_complex_wishart_10_r3_a0.99.mat`
- `denoised_complex_wishart_30_r3_a0.99.mat`
- `denoised_complex_wishart_50_r3_a0.99.mat`

각 파일에는:

- complex denoised image
- sigma map
- retained component map
- evaluation-only brain mask
- noise / acquisition metadata

가 저장된다.


In [ ]:
output_paths = {}

for noise_level in NOISE_LEVELS:
    output_path = (
        OUT_DIR /
        f"denoised_complex_wishart_{noise_level}_r{PATCH_R}_a{ALPHA}.mat"
    )

    if not output_path.exists():
        raise FileNotFoundError(
            f"{noise_level}% 결과 파일이 생성되지 않았습니다: {output_path}"
        )

    output_paths[noise_level] = output_path

    size_mb = output_path.stat().st_size / (1024 ** 2)

    print(
        f"{noise_level}% 결과 확인 | "
        f"{output_path.name} | {size_mb:.1f} MB"
    )


### 5. Magnitude 결과 시각화

Clean reference는 이 notebook에서 직접 사용하지 않는다.

각 noise level에 대해 다음을 확인한다.

- Noisy Magnitude
- Denoised Magnitude
- Removed = `Noisy - Denoised`

`Removed`는 최종 error map이 아니다.
Complex Wishart MP-PCA가 noisy magnitude에서 제거한 성분을 확인하기 위한 시각화이다.

최종 PSNR / SSIM / NRMSE / HFEN은 별도 평가 단계에서
clean reference와 `mask_brain`을 사용한다.


In [ ]:
echo_idx = 0

for noise_level in NOISE_LEVELS:
    noisy_npz = np.load(
        input_paths[noise_level],
        allow_pickle=False
    )
    noisy_complex = noisy_npz["noisy_complex"].astype(np.complex64)

    mat = sio.loadmat(output_paths[noise_level])
    deno_complex = mat["denoised_complex"].astype(np.complex64)

    if deno_complex.shape != noisy_complex.shape:
        raise ValueError(
            f"{noise_level}% visualization shape mismatch: "
            f"{deno_complex.shape} != {noisy_complex.shape}"
        )

    noisy_mag = np.abs(noisy_complex).astype(np.float32)
    deno_mag = np.abs(deno_complex).astype(np.float32)

    slice_idx = noisy_mag.shape[2] // 2

    noisy_2d = noisy_mag[:, :, slice_idx, echo_idx]
    deno_2d = deno_mag[:, :, slice_idx, echo_idx]
    removed_2d = noisy_2d - deno_2d

    vmin, vmax = np.percentile(noisy_2d, (1, 99))

    dv = float(np.percentile(np.abs(removed_2d), 99))
    dv = max(dv, 1e-8)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    ax[0].imshow(
        noisy_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[0].set_title(f"Noisy ({noise_level}%)")

    ax[1].imshow(
        deno_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[1].set_title("Complex Wishart MP-PCA")

    ax[2].imshow(
        removed_2d,
        cmap="bwr",
        vmin=-dv,
        vmax=dv
    )
    ax[2].set_title("Removed (Noisy - Denoised)")

    for a in ax:
        a.axis("off")

    fig.suptitle(
        f"3-Core Parallel Complex Wishart MP-PCA | "
        f"Noise {noise_level}% | Echo {echo_idx + 1} | Slice {slice_idx}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

    del noisy_complex, deno_complex, noisy_mag, deno_mag, mat


### 6. Sigma map / Retained component map 확인

Wishart MP-PCA가 추정한 `sigma_map`과 `ncomp_map`을 확인한다.

Denoising 자체는 full-volume에서 수행되지만,
향후 성능평가에서는 `mask_brain` 내부만 평가한다.


In [ ]:
for noise_level in NOISE_LEVELS:
    mat = sio.loadmat(output_paths[noise_level])

    sigma_map = np.asarray(mat["sigma_map"])
    ncomp_map = np.asarray(mat["ncomp_map"])

    slice_idx = sigma_map.shape[2] // 2

    fig, ax = plt.subplots(
        1,
        2,
        figsize=(10, 4)
    )

    im0 = ax[0].imshow(
        sigma_map[:, :, slice_idx]
    )
    ax[0].set_title(f"Sigma map ({noise_level}%)")
    ax[0].axis("off")
    plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

    im1 = ax[1].imshow(
        ncomp_map[:, :, slice_idx],
        vmin=0,
        vmax=N_ECHO
    )
    ax[1].set_title(f"Retained components ({noise_level}%)")
    ax[1].axis("off")
    plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    del mat, sigma_map, ncomp_map


### 7. 메모리 정리

병렬 worker는 각자 종료되면서 메모리가 반환된다.
부모 프로세스에서도 남은 참조를 정리한다.


In [ ]:
gc.collect()

print("07_Complex_Wishart_3 완료")
print("Output directory:", OUT_DIR)
print("Parallel workers:", N_WORKERS)
print("Patch radius:", PATCH_R)
print("Alpha:", ALPHA)
